# Whale Calls — ResNet18/BEATs Training (Colab)
Before running: set **Runtime → Change runtime type → GPU**.

In [ ]:
MODEL_ARCH = "resnet18"  # beats

In [ ]:
import torch

assert (
    torch.cuda.is_available()
), "No GPU found — go to Runtime → Change runtime type → GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
!git clone https://github.com/shaishechter/whale_calls.git /content/whale_calls
%cd /content/whale_calls
!pip install -r requirements.txt -q

In [ ]:
# using a local path is faster for training but requires re-downloading at every session
# we can be smarter about this in the future by caching the data in a shared location and symlinking/rsyncing it to the local path
DATA_PATH = "/content/whale_data"

In [ ]:
import os
from google.colab import drive

drive.mount("/content/drive")

LOGS_PATH = "/content/drive/MyDrive/whale_calls_logs"
os.makedirs(LOGS_PATH, exist_ok=True)

In [ ]:
import os
from google.colab import userdata

In [ ]:
import wandb

# WANDB_API_KEY should be set in Colab's User Settings, if not, login will prompt for credentials
wandb_api = userdata.get("WANDB_API_KEY")
if wandb_api is not None:
    os.environ["WANDB_API_KEY"] = wandb_api
else:
    wandb.login()

In [ ]:
from google.colab import userdata
import os

os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

In [ ]:
from hydra import compose, initialize_config_dir
from hydra.core.global_hydra import GlobalHydra
from omegaconf import OmegaConf

GlobalHydra.instance().clear()  # safe to re-run this cell

with initialize_config_dir(config_dir="/content/whale_calls/conf", version_base=None):
    cfg = compose(
        config_name=MODEL_ARCH,
        overrides=[
            f"data_base_path={DATA_PATH}",
            f"logs_base_path={LOGS_PATH}",
        ],
    )

print(OmegaConf.to_yaml(cfg))

In [ ]:
from hydra import compose, initialize_config_dir
from hydra.core.global_hydra import GlobalHydra
from omegaconf import OmegaConf

GlobalHydra.instance().clear()  # safe to re-run this cell

with initialize_config_dir(config_dir="/content/whale_calls/conf", version_base=None):
    cfg = compose(
        config_name=MODEL_ARCH,
        overrides=[f"data_base_path={DATA_PATH}"],
    )

print(OmegaConf.to_yaml(cfg))

In [ ]:
from hydra.utils import instantiate

datamodule = instantiate(cfg.datamodule)
datamodule.setup()

In [ ]:
model = instantiate(cfg.model)

logger = instantiate(cfg.trainer.logger)
logger.log_hyperparams(model.hparams)

trainer = instantiate(cfg.trainer, logger=logger)
trainer.fit(model, datamodule=datamodule)